# 02 - Feature Engineering

Building predictive features from raw data using domain-specific generators.

**Feature sets created:**
- Player performance features (rolling averages, value metrics, consistency)
- Market features (line movement, implied probability, CLV, EV)
- User behavioral features (engagement, financial, risk indicators)

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
from src.data.ingestion import DataLoader
from src.data.preprocessing import DataCleaner
from src.features.player_features import PlayerFeatureGenerator
from src.features.market_features import MarketFeatureGenerator
from src.features.user_features import UserFeatureGenerator
from src.features.feature_store import FeatureStore
from src.visualization.plots import PlotFactory

print('Setup complete')

In [ ]:
loader = DataLoader('../data')
cleaner = DataCleaner()

players = loader.get_sample_data('players')
users = loader.get_sample_data('user_profiles')
entries = loader.get_sample_data('user_entries')
market = loader.get_sample_data('market_odds')

## Player Feature Engineering

In [ ]:
pfg = PlayerFeatureGenerator(window_sizes=[3, 5, 10])
player_features = pfg.generate_all(players)

new_cols = set(player_features.columns) - set(players.columns)
print(f'Original columns: {len(players.columns)}')
print(f'After feature engineering: {len(player_features.columns)}')
print(f'New features ({len(new_cols)}): {sorted(new_cols)}')

In [ ]:
# Value analysis: best points per dollar
if 'points_per_dollar' in player_features.columns:
    top_value = player_features.nlargest(10, 'points_per_dollar')[
        ['name', 'position', 'salary', 'fantasy_points', 'points_per_dollar', 'value_score']
    ]
    print('Top 10 Value Players:')
    print(top_value.to_string(index=False))

In [ ]:
if 'points_per_dollar' in player_features.columns:
    fig = PlotFactory.scatter_plot(player_features, 'salary', 'fantasy_points',
                                    color='salary_tier', title='Salary Tiers vs Fantasy Points')
    fig.show()

## Market Feature Engineering

In [ ]:
mfg = MarketFeatureGenerator()
market_features = mfg.generate_all(market)

new_market_cols = set(market_features.columns) - set(market.columns)
print(f'New market features ({len(new_market_cols)}): {sorted(new_market_cols)}')

market_features[['line_movement', 'implied_probability', 'expected_value', 'clv']].describe().round(4)

In [ ]:
if 'ev_positive' in market_features.columns:
    ev_rate = market_features['ev_positive'].mean()
    print(f'Percentage of markets with positive EV: {ev_rate:.1%}')

if 'beat_closing_line' in market_features.columns:
    clv_rate = market_features['beat_closing_line'].mean()
    print(f'Percentage beating closing line: {clv_rate:.1%}')

## User Feature Engineering

In [ ]:
ufg = UserFeatureGenerator()
user_features = ufg.generate_all(users, entries)

new_user_cols = set(user_features.columns) - set(users.columns)
print(f'New user features ({len(new_user_cols)}): {sorted(new_user_cols)}')

if 'risk_level' in user_features.columns:
    print('\nRisk Level Distribution:')
    print(user_features['risk_level'].value_counts())

In [ ]:
if 'roi' in user_features.columns:
    fig = PlotFactory.distribution_plot(user_features['roi'].clip(-1, 1), 'User ROI Distribution')
    fig.show()

## Saving to Feature Store

In [ ]:
store = FeatureStore('../data/features')

# Save numeric-only feature sets for modeling
player_numeric = player_features.select_dtypes(include=['number'])
store.save_features('player_features', player_numeric, 'Player performance features v1')

user_numeric = user_features.select_dtypes(include=['number'])
store.save_features('user_features', user_numeric, 'User behavioral features v1')

market_numeric = market_features.select_dtypes(include=['number'])
store.save_features('market_features', market_numeric, 'Market odds features v1')

print('Feature sets saved:')
for name, info in store.list_feature_sets().items():
    print(f'  {name}: shape={info["shape"]}')

## Summary

| Feature Set | Original | After Engineering | New Features |
|-------------|----------|-------------------|--------------|
| Players | 16 | ~45 | Rolling avgs, value metrics, ranks |
| Market | 10 | ~20 | Line movement, implied prob, EV, CLV |
| Users | 13 | ~25 | ROI, engagement, risk indicators |

**Next:** `03_model_development.ipynb`